# Principles of Natural Language Processing Lab  
## Language Modelling: N-grams, N-gram Probabilities, Evaluation and Perplexity

**Course:** AME 5053 — Principles of Natural Language Processing Lab  
**Lab duration:** 3 hours  
**Mode:** Guided implementation + comparison + interpretation

### Learning outcomes
By the end of this lab, you should be able to:

1. construct unigram, bigram and trigram language models from a text corpus;
2. estimate N-gram probabilities using maximum-likelihood estimation (MLE);
3. compute the probability of a sentence under an N-gram model;
4. explain the effect of sentence-boundary symbols such as `<s>` and `</s>`;
5. evaluate a language model on unseen text using perplexity;
6. diagnose why an unsmoothed N-gram model can fail on unseen N-grams.

### What you must submit
Your notebook should contain:
- your predictions before running selected cells;
- completed implementations;
- at least one comparison between models;
- an error analysis using concrete examples;
- a short conclusion explaining what model behaviour you observed.

## 1. Corpus used in this lab

We will use a small, controlled corpus so that the counts can be inspected manually.

```text
students learn natural language processing
students learn machine learning
students study language models
language models predict words
language models assign probabilities
machine learning models learn patterns
natural language processing uses language models
```

The corpus is intentionally small. This makes the probability calculations transparent, but it will also expose an important limitation: **many valid word sequences will never occur in the training corpus**.

We will later split examples into **training** and **test** sentences so that perplexity is computed on data that the model did not simply memorize.

In [ ]:
from collections import Counter, defaultdict
import math
import pandas as pd

raw_sentences = [
    "students learn natural language processing",
    "students learn machine learning",
    "students study language models",
    "language models predict words",
    "language models assign probabilities",
    "machine learning models learn patterns",
    "natural language processing uses language models",
]

raw_sentences

## 1.1 Tokenize the corpus
Run the helper below and inspect the output.

In [ ]:
def tokenize(sentence):
    return sentence.lower().split()

tokenized = [tokenize(s) for s in raw_sentences]
tokenized

## 2. Sentence boundaries

A language model should know not only which words can follow other words, but also where a sentence can begin and end.

For a bigram model we will represent:

```text
students learn machine learning
```

as

```text
<s> students learn machine learning </s>
```

For a trigram model, we need enough history at the beginning. We therefore use two start symbols:

```text
<s> <s> students learn machine learning </s>
```

This convention allows the model to estimate probabilities for sentence beginnings as well as sentence endings.

### Prediction 1 — before coding

Without running any code, answer:

1. Which word do you expect to be the most frequent unigram?
2. Which bigram do you expect to have a relatively high probability?
3. Do you expect a trigram model to assign non-zero probability to more or fewer unseen test sentences than a bigram model? Why?

Write your answers below.

**Your prediction:**  
- Most frequent unigram:  
- Likely high-probability bigram:  
- Bigram vs trigram on unseen text:

## 3. Build N-gram counts

For an N-gram model, we count neighbouring sequences of words.

For example, in:

```text
language models predict words
```

the bigrams are:

```text
(<s>, language)
(language, models)
(models, predict)
(predict, words)
(words, </s>)
```

The trigram model uses sequences of three tokens.

We will use these counts to estimate conditional probabilities.

In [ ]:
def add_boundaries(tokens, n):
    # TODO:
    # Bigram model: add one <s> and one </s>
    # Trigram model: add two <s> symbols and one </s>
    pass

In [ ]:
def make_ngrams(tokens, n):
    # TODO: return all adjacent n-grams as tuples
    # Example: ['a','b','c'], n=2 -> [('a','b'), ('b','c')]
    pass

In [ ]:
# TODO: build unigram, bigram and trigram counts over the whole corpus.
# Suggested names:
# unigram_counts
# bigram_counts
# trigram_counts

unigram_counts = None
bigram_counts = None
trigram_counts = None

### Inspect the most frequent N-grams
Create tables showing the 10 most frequent unigrams, bigrams and trigrams.

In [ ]:
# TODO: display the 10 most frequent items from each Counter.

## 4. Maximum-likelihood estimation of N-gram probabilities

For a bigram model,

$$
P(w_i \mid w_{i-1})
=
\frac{C(w_{i-1}, w_i)}{C(w_{i-1})}
$$

For a trigram model,

$$
P(w_i \mid w_{i-2}, w_{i-1})
=
\frac{C(w_{i-2}, w_{i-1}, w_i)}
     {C(w_{i-2}, w_{i-1})}
$$

The key idea is simple: **among all occurrences of the history, how often was the next word the one we are asking about?**

This is a maximum-likelihood estimate because the probabilities are obtained directly from observed frequencies in the training corpus.

In [ ]:
def bigram_probability(w1, w2, bigram_counts, unigram_context_counts):
    # TODO: implement P(w2 | w1)
    pass

def trigram_probability(w1, w2, w3, trigram_counts, bigram_context_counts):
    # TODO: implement P(w3 | w1,w2)
    pass

### Probability checks
Compute and interpret the following:

- `P(learn | students)`
- `P(models | language)`
- `P(language | learn)`
- `P(models | study, language)`

In [ ]:
# TODO: create the context counts needed by the probability functions,
# then compute the four probabilities above.

## 5. Sentence probability

Under a bigram model, the probability of a sentence is approximated using the Markov assumption:

\[
P(w_1,\ldots,w_m)
\approx
P(w_1\mid <s>)
\prod_{i=2}^{m}P(w_i\mid w_{i-1})
P(</s>\mid w_m)
\]

The trigram model conditions each word on the previous two tokens.

Because sentence probabilities are products of many values smaller than 1, they can become extremely small. In larger systems we therefore often work with **log probabilities**.

In [ ]:
def bigram_sentence_probability(sentence, bigram_counts, context_counts):
    # TODO:
    # 1. tokenize
    # 2. add boundaries
    # 3. multiply bigram probabilities
    pass

def trigram_sentence_probability(sentence, trigram_counts, context_counts):
    # TODO:
    # 1. tokenize
    # 2. add trigram boundaries
    # 3. multiply trigram probabilities
    pass

### Compare sentence probabilities
Evaluate these two training-like sentences:

- `students learn machine learning`
- `language models assign probabilities`

Then explain why their probabilities differ.

In [ ]:
# TODO: compute bigram and trigram sentence probabilities for the two sentences.

## 6. Evaluation on unseen text

A useful language model should perform well on text that was **not used to estimate its probabilities**.

We will use:

### Test sentence A
`students learn language models`

### Test sentence B
`language models learn patterns`

### Test sentence C
`students predict probabilities`

Before computing anything, inspect the corpus and predict which test sentences may contain unseen bigrams or trigrams.

### Prediction 2 — unseen sequences

Before running the next section, list at least one bigram/trigram that you think is unseen in each test sentence.

**A. students learn language models**  
Prediction:

**B. language models learn patterns**  
Prediction:

**C. students predict probabilities**  
Prediction:

## 7. Perplexity

Perplexity is a standard intrinsic evaluation measure for language models.

For a sequence containing \(N\) predicted tokens,

\[
PP(W)
=
P(W)^{-1/N}
\]

Equivalently, using log probabilities,

\[
PP(W)
=
\exp\left(
-\frac{1}{N}
\sum_{i=1}^{N}\log P(w_i \mid \text{history})
\right)
\]

Interpretation:

- **lower perplexity** means the model assigns higher probability to the observed test sequence;
- **higher perplexity** means the sequence is more surprising to the model;
- if an unsmoothed model encounters an N-gram with probability zero, the perplexity becomes infinite.

Perplexity should only be compared meaningfully when models are evaluated on the **same tokenization and same test data**.

In [ ]:
def bigram_perplexity(sentence, bigram_counts, context_counts):
    # TODO:
    # Use log probabilities.
    # If any required bigram has probability 0, return math.inf.
    pass

def trigram_perplexity(sentence, trigram_counts, context_counts):
    # TODO:
    # Use log probabilities.
    # If any required trigram has probability 0, return math.inf.
    pass

In [ ]:
test_sentences = [
    "students learn language models",
    "language models learn patterns",
    "students predict probabilities",
]

# TODO: construct a DataFrame with:
# sentence | bigram_perplexity | trigram_perplexity

## 8. Diagnose the model
For every test sentence with infinite perplexity, print the first unseen N-gram that causes the failure.

In [ ]:
def first_unseen_ngram(sentence, n, ngram_counts):
    # TODO: return the first unseen n-gram, or None if every n-gram was seen.
    pass

# TODO: apply this to the test sentences for n=2 and n=3.

## 9. Interpretation and error analysis

Answer in complete sentences.

1. Which model gave lower perplexity on the test sentences that both models could score?
2. Did the higher-order model always perform better? Explain using the size of this corpus.
3. Identify one unseen bigram and one unseen trigram from the test set.
4. Why does a zero probability create a serious problem when sentence probability is computed by multiplication?
5. What would you expect to happen if the training corpus became much larger?

## 10. Viva / discussion questions

1. Why is an N-gram language model called a probabilistic model?
2. What is the Markov assumption in a bigram model?
3. Why are sentence boundary symbols useful?
4. Why can a trigram model be more data-hungry than a bigram model?
5. Why is perplexity usually preferred over raw sentence probability when comparing sequences of different lengths?
6. Why can an unsmoothed N-gram model assign infinite perplexity to a perfectly grammatical sentence?